In [17]:
import os
import random
from pathlib import Path
from urllib.request import urlretrieve
import requests
import dagshub
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ViT_B_16_Weights
from tqdm.auto import tqdm

In [18]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)


seed_everything(42)

In [22]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 16
    num_workers = 0
    epochs = 5
    lr = 1e-4
    weight_decay = 1e-5
    val_size = 0.2
    pretrained = False
    freeze_backbone = False
    unfreeze_last_blocks = 0
    dropout = 0.3
    hidden_size = 256
    artifacts_dir = "model_outputs/ViT-v0",
    model_type = 'ViT-v0'
    model_name = "ViT-v0.pt"
    experiment_name = "amazon_smart_pricing"
    run_name = "ViT-v0-no-pretrain" # название на сайте

In [23]:
dagshub.init(
    repo_owner="Dezurn",
    repo_name="Deep-Learning",
    mlflow=True,
)

mlflow.set_experiment(CFG.experiment_name)

Initialized MLflow to track repo "Dezurn/Deep-Learning"

Repository Dezurn/Deep-Learning initialized!

<Experiment: artifact_location='mlflow-artifacts:/99b0aadd413f4a00bf17d371a1a7a7aa', creation_time=1781452541346, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781452541346, lifecycle_stage='active', name='amazon_smart_pricing', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [24]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")
train_df.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [25]:
num = 5000
work_df = train_df.sample(num, random_state=CFG.seed).reset_index(drop=True)
val_data = work_df.iloc[: int(len(work_df) * CFG.val_size)].reset_index(drop=True)
train_data = work_df.iloc[int(len(work_df) * CFG.val_size) :].reset_index(drop=True)

In [26]:
len(val_data), len(train_data)

(1000, 4000)

In [27]:
from urllib.request import urlretrieve
from pathlib import Path
from tqdm.auto import tqdm
def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []

    for url in tqdm(df[CFG.image_col], desc="Downloading images"):
        filename = str(url).split("/")[-1].split("?")[0]
        image_path = f"{CFG.cache_dir}/{filename}"

        try:
            if not Path(image_path).exists():
                urlretrieve(url, image_path)

            image_paths.append(image_path)

        except Exception:
            image_paths.append(None)

    df = df.copy()
    df["image_path"] = image_paths
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [28]:
import requests
from pathlib import Path
from tqdm.auto import tqdm

def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []
    errors = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0.0.0 Safari/537.36"
        )
    }

    for url in tqdm(df[CFG.image_col], desc="Downloading images"):
        filename = str(url).split("/")[-1].split("?")[0]
        image_path = Path(CFG.cache_dir) / filename

        try:
            if not image_path.exists():
                response = requests.get(url, headers=headers, timeout=20)
                response.raise_for_status()

                with open(image_path, "wb") as f:
                    f.write(response.content)

            image_paths.append(str(image_path))

        except Exception as e:
            image_paths.append(None)
            errors.append((url, str(e)))

    df = df.copy()
    df["image_path"] = image_paths

    print("Всего строк:", len(df))
    print("Скачано/найдено картинок:", df["image_path"].notna().sum())
    print("Ошибок:", len(errors))

    if errors:
        print("Пример ошибки:")
        print(errors[0])

    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [13]:
train_data = download_images_simple(train_data)
val_data = download_images_simple(val_data)

In [29]:
train_data.head(), val_data.head()

(   sample_id                                    catalog_content  \
 0     156787  Item Name: 'Rich JW Allen Snow Queen Icing Bas...   
 1     225070  Item Name: Dandies Vegan Marshmallows, Vanilla...   
 2      46913  Item Name: Canels Gum Box Original 60 Count\r\...   
 3     298119  Item Name: HERSHEY'S Miniatures Assorted Choco...   
 4      68001  Item Name: Pink Lemonade Licorice Sour Sticks ...   
 
                                           image_link    price  
 0  https://m.media-amazon.com/images/I/71mDwF1ANo...  113.310  
 1  https://m.media-amazon.com/images/I/71hAUfz93S...    5.490  
 2  https://m.media-amazon.com/images/I/91DIhiMcqI...    4.035  
 3  https://m.media-amazon.com/images/I/51iYRznRks...   20.790  
 4  https://m.media-amazon.com/images/I/91hRyLkNi+...   18.990  ,
    sample_id                                    catalog_content  \
 0     158784  Item Name: Log Cabin Sugar Free Syrup, 24 FL O...   
 1       4095  Item Name: Raspberry Ginseng Oolong Tea (50 te..

In [30]:
class ImagePriceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        price = np.log1p(row[CFG.target_col]).astype("float32")

        return image, torch.tensor(price, dtype=torch.float32)

In [31]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = T.Compose([
    T.Resize(256), # меняем размер
    T.RandomCrop(CFG.img_size), # вырезаем случайную область указанного размера
    T.RandomHorizontalFlip(), # зеркальное отражение по горизонтали
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD) # нормализация
])

valid_transform = T.Compose([ # на валидации нельзя делать никаких преобразований (кроме изменения размера),
                               # ведь мы должны проверить модель на реальных картинках
    T.Resize(256),
    T.CenterCrop(CFG.img_size),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [32]:
train_dataset = ImagePriceDataset(train_data, transform=train_transform)
val_dataset = ImagePriceDataset(val_data, transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)

val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Дальше модель

In [34]:
class ViTPriceRegressor(nn.Module):
    def __init__(
        self,
        pretrained=False,
        freeze_backbone=False,
        unfreeze_last_blocks=0,
        hidden_size=256,
        dropout=0.3
    ):
        super().__init__()

        if pretrained:
            weights = ViT_B_16_Weights.DEFAULT
            self.model = models.vit_b_16(weights=weights)
        else:
            self.model = models.vit_b_16(weights=None)

        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False

            for param in self.model.heads.parameters():
                param.requires_grad = True

            if unfreeze_last_blocks > 0:
                for block in self.model.encoder.layers[-unfreeze_last_blocks:]:
                    for param in block.parameters():
                        param.requires_grad = True

                for param in self.model.encoder.ln.parameters():
                    param.requires_grad = True

        in_features = self.model.heads.head.in_features

        self.model.heads.head = nn.Sequential(
            nn.Linear(in_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

In [35]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_vit.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [13]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [36]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0

    preds = []
    targets_all = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)

            preds.extend(outputs.detach().cpu().numpy())
            targets_all.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    preds = np.array(preds)
    targets_all = np.array(targets_all)

    preds_price = np.expm1(preds)
    targets_price = np.expm1(targets_all)

    preds_price = np.clip(preds_price, 0, None)

    mae = mean_absolute_error(targets_price, preds_price)
    mse = mean_squared_error(targets_price, preds_price)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_price, preds_price)

    metrics = {
        "val_loss": epoch_loss,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

    return metrics

In [37]:
def fit_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device):
    best_rmse = float("inf")
    history = []

    os.makedirs(CFG.artifact_dir, exist_ok=True)
    best_model_path = os.path.join(CFG.artifact_dir, CFG.model_name)
    history_path = os.path.join(CFG.artifact_dir, "vit_history.csv")

    mlflow.log_params(
        {
            "model": CFG.model_type,
            "pretrained": True,
            "frozen_backbone": True,
            "target": "log1p(price)",
            "img_size": CFG.img_size,
            "batch_size": CFG.batch_size,
            "epochs": CFG.epochs,
            "lr": CFG.lr,
            "weight_decay": CFG.weight_decay,
            "optimizer": "AdamW",
            "loss": "MSELoss",
            "scheduler": "ReduceLROnPlateau",
            "val_size": CFG.val_size,
            "seed": CFG.seed,
        }
    )

    for epoch in range(1, CFG.epochs + 1):
        print(f"\nEpoch {epoch}/{CFG.epochs}")

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )

        val_metrics = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
        )

        scheduler.step(val_metrics["val_loss"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["val_loss"],
            "mae": val_metrics["mae"],
            "rmse": val_metrics["rmse"],
            "r2": val_metrics["r2"],
        }

        history.append(row)

        mlflow.log_metrics(row, step=epoch)

        print(
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_metrics['val_loss']:.4f} | "
            f"MAE: {val_metrics['mae']:.2f} | "
            f"RMSE: {val_metrics['rmse']:.2f} | "
            f"R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_rmse": best_rmse,
                },
                best_model_path,
            )

            print(f"Best model saved: {best_model_path}")

    history = pd.DataFrame(history)
    history.to_csv(history_path, index=False)
    
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint["model_state_dict"])

    mlflow.log_metric("best_rmse", best_rmse)
    mlflow.log_artifact(best_model_path, artifact_path="model_checkpoint")
    mlflow.log_artifact(history_path, artifact_path="history")
    mlflow.pytorch.log_model(model, artifact_path="model")

    return model, history

In [38]:
with mlflow.start_run(run_name=CFG.run_name):
    model_vit, history_vit = fit_model(
        model=model_vit,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )

🏃 View run ViT-v0-no-pretrain at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/07666f2530db4528acd299c0e1755c80
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1


AttributeError: type object 'CFG' has no attribute 'artifact_dir'